# Smart HVAC Energy Analytics & Predictive Maintenance System

# Smart HVAC Energy Analytics & Predictive Maintenance System

## Phase 2 — Data Preprocessing

This notebook prepares the LBNL RTU dataset for downstream machine learning tasks.

The preprocessing workflow includes:

- Discovering healthy and faulty HVAC datasets
- Extracting fault labels and severity from filenames
- Combining the datasets into a master dataset
- Separating simulation and site datasets
- Checking data quality
- Converting timestamps
- Removing columns that are specific to the other dataset type
- Saving the processed datasets for subsequent analysis and modeling

### 1. Import Libraries

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATASET_DIR = PROJECT_ROOT / "datasets"

LBNL_PATH = DATASET_DIR / "LBNL_RTU"
GENOME_PATH = DATASET_DIR / "Building_Genome"

OUTPUT_PATH = PROJECT_ROOT / "outputs"

OUTPUT_PATH.mkdir(exist_ok=True)

## 2. Discover Healthy and Faulty RTU Datasets

The LBNL RTU data is organized into healthy and faulty operating conditions.

The filenames contain useful information about the fault type and severity, which will be extracted in the next section.

In [3]:
healthy_files = list((LBNL_PATH / "Healthy").glob("*.csv"))
fault_files = list((LBNL_PATH / "faults").glob("*.csv"))

print(len(healthy_files))
print(len(fault_files))

3
14


## 3. Extract Health Status, Fault Type, and Severity

The dataset filenames encode the operating condition.

For example:

- `baseline` / `unfaulted` → Healthy
- `condfouling20` → Condenser Fouling, severity 20
- `evapfouling30` → Evaporator Fouling, severity 30
- `overcharge15` → Overcharge, severity 15
- `undercharge20` → Undercharge, severity 20
- `staging` → Staging Fault

These labels are added to every sensor record so that the dataset can be used for supervised machine learning.

In [12]:
def extract_labels(filename):
    """
    Extract health status, fault type, and severity from filename.
    """
    
    name = filename.lower()

    if "baseline" in name or "unfaulted" in name:
        return "Healthy", "Healthy", 0

    if "condfouling" in name:
        severity = int(re.findall(r"\d+", name)[0])
        return "Fault", "Condenser Fouling", severity

    if "evapfouling" in name:
        severity = int(re.findall(r"\d+", name)[0])
        return "Fault", "Evaporator Fouling", severity

    if "overcharge" in name:
        severity = int(re.findall(r"\d+", name)[0])
        return "Fault", "Overcharge", severity

    if "undercharge" in name:
        severity = int(re.findall(r"\d+", name)[-1])
        return "Fault", "Undercharge", severity

    if "staging" in name:
        return "Fault", "Staging Fault", 100

    return "Unknown", "Unknown", -1

test_files = [
    "RTU_sim_condfouling20.csv",
    "RTU_sim_undercharge15.csv",
    "RTU_sim_baseline.csv",
    "Site1_Staging_Fault.csv",
    "Site2_Unfaulted.csv"]

for f in test_files:
    print(f, "->", extract_labels(f))

RTU_sim_condfouling20.csv -> ('Fault', 'Condenser Fouling', 20)
RTU_sim_undercharge15.csv -> ('Fault', 'Undercharge', 15)
RTU_sim_baseline.csv -> ('Healthy', 'Healthy', 0)
Site1_Staging_Fault.csv -> ('Fault', 'Staging Fault', 100)
Site2_Unfaulted.csv -> ('Healthy', 'Healthy', 0)


# 4. Build the Master LBNL Dataset

All healthy and faulty CSV files are loaded and combined into a single master dataset.

Three additional target-related columns are added:

- `health_status`
- `fault_type`
- `severity`

This creates a unified dataset for subsequent preprocessing and model development.

In [13]:
lbnl_dataframes = []

for file in healthy_files:

    df = pd.read_csv(file)

    health_status, fault_type, severity = extract_labels(file.name)

    df["health_status"] = health_status
    df["fault_type"] = fault_type
    df["severity"] = severity

    lbnl_dataframes.append(df)

In [14]:
for file in fault_files:

    df = pd.read_csv(file)

    health_status, fault_type, severity = extract_labels(file.name)

    df["health_status"] = health_status
    df["fault_type"] = fault_type
    df["severity"] = severity

    lbnl_dataframes.append(df)

In [15]:
lbnl_master = pd.concat(
    lbnl_dataframes,
    ignore_index=True
)

In [16]:
print(lbnl_master.shape)

lbnl_master.head()

(2332649, 41)


,Datetime,RTU_COMP_WATT,RTU_OA_FLOW,RTU_OA_HUM,RTU_OA_TEMP,RTU_RA_FLOW,RTU_RA_HUM,RTU_RA_TEMP,RTU_REFG_COND_PRES,RTU_REFG_COND_TEMP,...,RTU_REFG_COND_TEMP_1,RTU_REFG_COND_TEMP_2,RTU_REFG_DISC_PRES_1,RTU_REFG_DISC_PRES_2,RTU_REFG_DISC_TEMP_1,RTU_REFG_DISC_TEMP_2,RTU_REFG_SUCT_PRES_1,RTU_REFG_SUCT_PRES_2,RTU_REFG_SUCT_TEMP_1,RTU_REFG_SUCT_TEMP_2
0,2018-07-20 01:00:00,2097.6797,181.93083,49.361565,55.040030,2941.1106,50.000000,75.200000,24763394.0,79.043396,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-07-20 01:01:00,2089.0981,181.93083,59.133976,55.033990,2941.1106,59.898830,73.353580,24861452.0,79.312790,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-07-20 01:02:00,2089.8665,181.93083,59.105840,55.028000,2941.1106,59.870330,73.096176,24846182.0,79.270930,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-07-20 01:03:00,2091.4324,181.93083,59.077980,55.022015,2941.1106,59.842117,72.839096,24822794.0,79.206710,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-07-20 01:04:00,2093.2980,181.93083,59.050636,55.016030,2941.1106,59.814423,72.583115,24796212.0,79.133650,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Inspect Dataset Labels

The following distributions confirm how healthy and faulty operating conditions are represented in the combined dataset.

In [17]:
lbnl_master[["health_status","fault_type","severity"]].sample(10)

,health_status,fault_type,severity
1287840,Fault,Evaporator Fouling,30
1235486,Fault,Evaporator Fouling,30
2026178,Fault,Undercharge,15
1689749,Fault,Overcharge,20
370154,Healthy,Healthy,0
71334,Healthy,Healthy,0
1228433,Fault,Evaporator Fouling,30
2204702,Fault,Undercharge,20
1993790,Fault,Undercharge,15
641660,Fault,Condenser Fouling,20


In [18]:
print(lbnl_master["health_status"].value_counts())

print()

print(lbnl_master["fault_type"].value_counts())

print()

print(lbnl_master["severity"].value_counts().sort_index())

health_status
Fault      1855168
Healthy     477481
Name: count, dtype: int64

fault_type
Healthy               477481
Undercharge           473345
Condenser Fouling     431823
Evaporator Fouling    431823
Overcharge            431823
Staging Fault          86354
Name: count, dtype: int64

severity
0      477481
10     575764
15     287882
20     575764
30     287882
40      41522
100     86354
Name: count, dtype: int64


In [19]:
print(f"Total Columns: {len(lbnl_master.columns)}")

for i, col in enumerate(lbnl_master.columns, start=1):
    print(f"{i:2d}. {col}")

Total Columns: 41
 1. Datetime
 2. RTU_COMP_WATT
 3. RTU_OA_FLOW
 4. RTU_OA_HUM
 5. RTU_OA_TEMP
 6. RTU_RA_FLOW
 7. RTU_RA_HUM
 8. RTU_RA_TEMP
 9. RTU_REFG_COND_PRES
10. RTU_REFG_COND_TEMP
11. RTU_REFG_DISC_PRES
12. RTU_REFG_DISC_TEMP
13. RTU_REFG_SUCT_PRES
14. RTU_REFG_SUCT_TEMP
15. RTU_SA_FAN_WATT
16. RTU_SA_FLOW
17. RTU_SA_HUM
18. RTU_SA_TEMP
19. RTU_SEN_CAPA
20. RTU_STG_STA
21. RTU_TOT_CAPA
22. RTU_TOT_WATT
23. ZA_HUM
24. ZA_TEMP
25. ZA_TEMP_SPT
26. health_status
27. fault_type
28. severity
29. RTU_LA_COND_TEMP
30. RTU_MA_HUM
31. RTU_MA_TEMP
32. RTU_REFG_COND_TEMP_1
33. RTU_REFG_COND_TEMP_2
34. RTU_REFG_DISC_PRES_1
35. RTU_REFG_DISC_PRES_2
36. RTU_REFG_DISC_TEMP_1
37. RTU_REFG_DISC_TEMP_2
38. RTU_REFG_SUCT_PRES_1
39. RTU_REFG_SUCT_PRES_2
40. RTU_REFG_SUCT_TEMP_1
41. RTU_REFG_SUCT_TEMP_2


# 6. Initial Data Quality Assessment

Before splitting the data into simulation and site datasets, the structure and missing-value patterns are examined.

This helps identify differences between the two data sources before applying dataset-specific preprocessing.

In [20]:
missing = (
    lbnl_master
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

print(missing)

RTU_MA_HUM              1871233
RTU_REFG_SUCT_PRES_1    1871233
RTU_REFG_SUCT_PRES_2    1871233
RTU_REFG_SUCT_TEMP_1    1871233
RTU_REFG_DISC_TEMP_2    1871233
RTU_REFG_SUCT_TEMP_2    1871233
RTU_REFG_DISC_TEMP_1    1871233
RTU_REFG_DISC_PRES_2    1871233
RTU_REFG_DISC_PRES_1    1871233
RTU_LA_COND_TEMP        1871233
RTU_MA_TEMP             1871233
RTU_REFG_COND_TEMP_1    1871233
RTU_REFG_COND_TEMP_2    1871233
RTU_SEN_CAPA             461416
ZA_TEMP_SPT              461416
RTU_RA_FLOW              461416
RTU_REFG_COND_PRES       461416
RTU_OA_FLOW              461416
RTU_STG_STA              461416
RTU_TOT_CAPA             461416
RTU_REFG_SUCT_TEMP       461416
RTU_REFG_SUCT_PRES       461416
RTU_REFG_COND_TEMP       461416
RTU_REFG_DISC_TEMP       461416
RTU_REFG_DISC_PRES       461416
RTU_RA_TEMP                   0
RTU_RA_HUM                    0
RTU_OA_HUM                    0
RTU_OA_TEMP                   0
RTU_COMP_WATT                 0
Datetime                      0
RTU_SA_H

In [21]:
for file in healthy_files + fault_files:
    df = pd.read_csv(file)

    print("=" * 70)
    print(file.name)
    print(f"Columns: {len(df.columns)}")
    print(df.columns.tolist())

RTU_sim_baseline.csv
Columns: 25
['Datetime', 'RTU_COMP_WATT', 'RTU_OA_FLOW', 'RTU_OA_HUM', 'RTU_OA_TEMP', 'RTU_RA_FLOW', 'RTU_RA_HUM', 'RTU_RA_TEMP', 'RTU_REFG_COND_PRES', 'RTU_REFG_COND_TEMP', 'RTU_REFG_DISC_PRES', 'RTU_REFG_DISC_TEMP', 'RTU_REFG_SUCT_PRES', 'RTU_REFG_SUCT_TEMP', 'RTU_SA_FAN_WATT', 'RTU_SA_FLOW', 'RTU_SA_HUM', 'RTU_SA_TEMP', 'RTU_SEN_CAPA', 'RTU_STG_STA', 'RTU_TOT_CAPA', 'RTU_TOT_WATT', 'ZA_HUM', 'ZA_TEMP', 'ZA_TEMP_SPT']
Site1_Unfaulted.csv
Columns: 26
['Datetime', 'RTU_COMP_WATT', 'RTU_LA_COND_TEMP', 'RTU_MA_HUM', 'RTU_MA_TEMP', 'RTU_OA_HUM', 'RTU_OA_TEMP', 'RTU_RA_HUM', 'RTU_RA_TEMP', 'RTU_REFG_COND_TEMP_1', 'RTU_REFG_COND_TEMP_2', 'RTU_REFG_DISC_PRES_1', 'RTU_REFG_DISC_PRES_2', 'RTU_REFG_DISC_TEMP_1', 'RTU_REFG_DISC_TEMP_2', 'RTU_REFG_SUCT_PRES_1', 'RTU_REFG_SUCT_PRES_2', 'RTU_REFG_SUCT_TEMP_1', 'RTU_REFG_SUCT_TEMP_2', 'RTU_SA_FAN_WATT', 'RTU_SA_FLOW', 'RTU_SA_HUM', 'RTU_SA_TEMP', 'RTU_TOT_WATT', 'ZA_HUM', 'ZA_TEMP']
Site2_Unfaulted.csv
Columns: 26
['Datetime', '

# 7. Separate Simulation and Site Datasets

The LBNL data contains two types of records:

- **Simulation data** — identified by the presence of `RTU_OA_FLOW`
- **Site data** — records where this variable is unavailable

The datasets are separated because they contain different sensor configurations and therefore require different feature handling.

In [24]:
processed_simulation = lbnl_master[
    lbnl_master["RTU_OA_FLOW"].notna()
].copy()

processed_site = lbnl_master[
    lbnl_master["RTU_OA_FLOW"].isna()
].copy()

In [25]:
print("Simulation Dataset:", processed_simulation.shape)
print("Site Dataset:", processed_site.shape)

Simulation Dataset: (1871233, 41)
Site Dataset: (461416, 41)


In [26]:
processed_simulation[
    ["RTU_OA_FLOW", "RTU_LA_COND_TEMP"]
].head()

,RTU_OA_FLOW,RTU_LA_COND_TEMP
0,181.93083,NaN
1,181.93083,NaN
2,181.93083,NaN
3,181.93083,NaN
4,181.93083,NaN


In [27]:
processed_site[
    ["RTU_OA_FLOW", "RTU_LA_COND_TEMP"]
].head()

,RTU_OA_FLOW,RTU_LA_COND_TEMP
143941,NaN,20.228651
143942,NaN,20.134947
143943,NaN,20.107876
143944,NaN,20.191168
143945,NaN,20.224487


In [28]:
processed_simulation.to_csv(
    OUTPUT_PATH / "processed_simulation.csv",
    index=False
)

processed_site.to_csv(
    OUTPUT_PATH / "processed_site.csv",
    index=False
)

In [29]:
def data_quality_report(df, name):

    print("="*60)
    print(name)
    print("="*60)

    print(f"Shape : {df.shape}")

    print("\nData Types")
    print(df.dtypes)

    print("\nMissing Values")
    print(df.isnull().sum()[df.isnull().sum() > 0])

    print("\nDuplicate Rows")
    print(df.duplicated().sum())

In [30]:
data_quality_report(
    processed_simulation,
    "Simulation Dataset"
)

data_quality_report(
    processed_site,
    "Site Dataset"
)

Simulation Dataset
Shape : (1871233, 41)

Data Types
Datetime                    str
RTU_COMP_WATT           float64
RTU_OA_FLOW             float64
RTU_OA_HUM              float64
RTU_OA_TEMP             float64
RTU_RA_FLOW             float64
RTU_RA_HUM              float64
RTU_RA_TEMP             float64
RTU_REFG_COND_PRES      float64
RTU_REFG_COND_TEMP      float64
RTU_REFG_DISC_PRES      float64
RTU_REFG_DISC_TEMP      float64
RTU_REFG_SUCT_PRES      float64
RTU_REFG_SUCT_TEMP      float64
RTU_SA_FAN_WATT         float64
RTU_SA_FLOW             float64
RTU_SA_HUM              float64
RTU_SA_TEMP             float64
RTU_SEN_CAPA            float64
RTU_STG_STA             float64
RTU_TOT_CAPA            float64
RTU_TOT_WATT            float64
ZA_HUM                  float64
ZA_TEMP                 float64
ZA_TEMP_SPT             float64
health_status               str
fault_type                  str
severity                  int64
RTU_LA_COND_TEMP        float64
RTU_MA_HUM         

# 8. Dataset-Specific Feature Cleaning

The simulation and site datasets contain different sensor configurations.

Features that are unavailable or specific to the other dataset type are removed so that each dataset contains only the relevant measurements.

In [31]:
processed_simulation["Datetime"] = pd.to_datetime(
    processed_simulation["Datetime"]
)

processed_site["Datetime"] = pd.to_datetime(
    processed_site["Datetime"]
)

In [32]:
print(processed_simulation.dtypes["Datetime"])
print(processed_site.dtypes["Datetime"])

datetime64[us]
datetime64[us]


# 10. Final Preprocessing Validation

The final datasets are checked for remaining missing values and their resulting dimensions before being saved.

At this stage, the simulation and site datasets contain their respective relevant sensor features.

In [33]:
missing_sim = processed_simulation.isnull().sum()

missing_sim[missing_sim > 0]

missing_site = processed_site.isnull().sum()

missing_site[missing_site > 0]

RTU_OA_FLOW           461416
RTU_RA_FLOW           461416
RTU_REFG_COND_PRES    461416
RTU_REFG_COND_TEMP    461416
RTU_REFG_DISC_PRES    461416
RTU_REFG_DISC_TEMP    461416
RTU_REFG_SUCT_PRES    461416
RTU_REFG_SUCT_TEMP    461416
RTU_SEN_CAPA          461416
RTU_STG_STA           461416
RTU_TOT_CAPA          461416
ZA_TEMP_SPT           461416
dtype: int64

In [34]:
print(processed_simulation.shape)
print(processed_site.shape)

(1871233, 41)
(461416, 41)


In [35]:
print(processed_simulation["RTU_OA_FLOW"].isna().sum())
print(processed_site["RTU_OA_FLOW"].isna().sum())

0
461416


In [36]:
missing_sim = processed_simulation.isnull().sum()
missing_sim[missing_sim > 0]

RTU_LA_COND_TEMP        1871233
RTU_MA_HUM              1871233
RTU_MA_TEMP             1871233
RTU_REFG_COND_TEMP_1    1871233
RTU_REFG_COND_TEMP_2    1871233
RTU_REFG_DISC_PRES_1    1871233
RTU_REFG_DISC_PRES_2    1871233
RTU_REFG_DISC_TEMP_1    1871233
RTU_REFG_DISC_TEMP_2    1871233
RTU_REFG_SUCT_PRES_1    1871233
RTU_REFG_SUCT_PRES_2    1871233
RTU_REFG_SUCT_TEMP_1    1871233
RTU_REFG_SUCT_TEMP_2    1871233
dtype: int64

In [37]:
missing_site = processed_site.isnull().sum()
missing_site[missing_site > 0]

RTU_OA_FLOW           461416
RTU_RA_FLOW           461416
RTU_REFG_COND_PRES    461416
RTU_REFG_COND_TEMP    461416
RTU_REFG_DISC_PRES    461416
RTU_REFG_DISC_TEMP    461416
RTU_REFG_SUCT_PRES    461416
RTU_REFG_SUCT_TEMP    461416
RTU_SEN_CAPA          461416
RTU_STG_STA           461416
RTU_TOT_CAPA          461416
ZA_TEMP_SPT           461416
dtype: int64

In [38]:
processed_simulation.isnull().sum().sum()
processed_site.isnull().sum().sum()

np.int64(5536992)

In [39]:
site_columns = [
    "RTU_LA_COND_TEMP",
    "RTU_MA_HUM",
    "RTU_MA_TEMP",
    "RTU_REFG_COND_TEMP_1",
    "RTU_REFG_COND_TEMP_2",
    "RTU_REFG_DISC_PRES_1",
    "RTU_REFG_DISC_PRES_2",
    "RTU_REFG_DISC_TEMP_1",
    "RTU_REFG_DISC_TEMP_2",
    "RTU_REFG_SUCT_PRES_1",
    "RTU_REFG_SUCT_PRES_2",
    "RTU_REFG_SUCT_TEMP_1",
    "RTU_REFG_SUCT_TEMP_2"
]

processed_simulation.drop(
    columns=site_columns,
    inplace=True,
    errors="ignore"
)

In [40]:
simulation_columns = [
    "RTU_OA_FLOW",
    "RTU_RA_FLOW",
    "RTU_REFG_COND_PRES",
    "RTU_REFG_COND_TEMP",
    "RTU_REFG_DISC_PRES",
    "RTU_REFG_DISC_TEMP",
    "RTU_REFG_SUCT_PRES",
    "RTU_REFG_SUCT_TEMP",
    "RTU_SEN_CAPA",
    "RTU_STG_STA",
    "RTU_TOT_CAPA",
    "ZA_TEMP_SPT"
]

processed_site.drop(
    columns=simulation_columns,
    inplace=True,
    errors="ignore"
)

In [41]:
print(processed_simulation.shape)
print(processed_site.shape)

print(processed_simulation.isnull().sum().sum())
print(processed_site.isnull().sum().sum())

(1871233, 28)
(461416, 29)
0
0


# 11. Conclusion

The LBNL RTU datasets were successfully preprocessed by:

- Identifying healthy and faulty operating conditions
- Extracting fault type and severity from filenames
- Combining all RTU records into a master dataset
- Separating simulation and site data
- Removing dataset-specific features
- Converting timestamps to datetime format
- Validating the cleaned datasets
- Saving the processed datasets for downstream machine learning tasks

The resulting datasets are now ready for exploratory analysis and feature engineering.